# Round 5 EDA — 50 products, 10 categories of 5

Position limit 10 PER product (small → rapid turnover required).
Three days of historical data: days 2/3/4.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from itertools import combinations
BASE = '/Users/markiejr/Propserity_4/data/ROUND5/ROUND_5'
frames = []
for d in [2,3,4]:
    df = pd.read_csv(f'{BASE}/prices_round_5_day_{d}.csv', sep=';')
    df['day'] = d; frames.append(df)
px = pd.concat(frames, ignore_index=True)
px['t_global'] = px['day']*1_000_000 + px['timestamp']
px = px.sort_values('t_global').reset_index(drop=True)
mid = px.pivot_table(index='t_global', columns='product', values='mid_price', aggfunc='first')
print('shape', mid.shape, 'products', mid.columns.size)

CATS = {
    'GALAXY_SOUNDS': ['DARK_MATTER','BLACK_HOLES','PLANETARY_RINGS','SOLAR_WINDS','SOLAR_FLAMES'],
    'SLEEP_POD':     ['SUEDE','LAMB_WOOL','POLYESTER','NYLON','COTTON'],
    'MICROCHIP':     ['CIRCLE','OVAL','SQUARE','RECTANGLE','TRIANGLE'],
    'PEBBLES':       ['XS','S','M','L','XL'],
    'ROBOT':         ['VACUUMING','MOPPING','DISHES','LAUNDRY','IRONING'],
    'UV_VISOR':      ['YELLOW','AMBER','ORANGE','RED','MAGENTA'],
    'TRANSLATOR':    ['SPACE_GRAY','ASTRO_BLACK','ECLIPSE_CHARCOAL','GRAPHITE_MIST','VOID_BLUE'],
    'PANEL':         ['1X2','2X2','1X4','2X4','4X4'],
    'OXYGEN_SHAKE':  ['MORNING_BREATH','EVENING_BREATH','MINT','CHOCOLATE','GARLIC'],
    'SNACKPACK':     ['CHOCOLATE','VANILLA','PISTACHIO','STRAWBERRY','RASPBERRY'],
}

## 1. Per-category basket sum: any constants?

In [ ]:
rows = []
for cat, names in CATS.items():
    cols = [f'{cat}_{n}' for n in names]
    s = mid[cols].sum(axis=1, skipna=False).dropna()
    rows.append({'cat': cat, 'mean_sum': s.mean(), 'std_sum': s.std(), 'min': s.min(), 'max': s.max()})
pd.DataFrame(rows).round(2)

**Top finding: PEBBLES sum = 50,000 EXACTLY** (std ≈ 2.8 — sub-tick noise).

**Second: SNACKPACK sum ≈ 50,221** (std ≈ 190 — soft constraint).

All other categories: std 700–2900 → no basket constraint. Independent.

## 2. Intra-category return correlations (basket structure)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, (cat, names) in zip(axes.flat, CATS.items()):
    cols = [f'{cat}_{n}' for n in names]
    rets = mid[cols].diff().dropna()
    corr = rets.corr()
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(5)); ax.set_xticklabels([n[:4] for n in names], rotation=45)
    ax.set_yticks(range(5)); ax.set_yticklabels([n[:4] for n in names])
    ax.set_title(cat, fontsize=10)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.tight_layout(); plt.show()

**Reading:**
- **PEBBLES**: XL has −0.50 with each of XS/S/M/L → signature of `XS+S+M+L+XL = const`. Confirmed.
- **SNACKPACK**: CHOCOLATE/VANILLA = −0.92 (paired anti-corr); PISTACHIO/STRAWBERRY = +0.91; STRAWBERRY/RASPBERRY = −0.92. Multi-product structure.
- All other categories: ~0 correlations. Independent products.

## 3. Pebbles basket deviation distribution

In [ ]:
P = mid[[f'PEBBLES_{n}' for n in CATS['PEBBLES']]].dropna()
dev = P.sum(axis=1) - 50000
print('Distribution of (sum − 50000):')
print(dev.describe())
print()
for t in [0.5, 1, 2, 5, 10, 15]:
    pos = (dev > t).sum(); neg = (dev < -t).sum()
    print(f'  |dev| > {t}: pos {pos} ({pos/len(dev)*100:.2f}%), neg {neg} ({neg/len(dev)*100:.2f}%)')
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(dev, bins=80); ax[0].set_title('PEBBLES sum − 50000 (histogram)')
ax[1].plot(dev.values, lw=0.3); ax[1].set_title('PEBBLES basket deviation over time')
plt.tight_layout(); plt.show()

**Reading:** 40% of ticks the basket is exactly 50,000. About 1.5% of ticks the deviation exceeds ±5 — these are the actionable arbs.

## 4. MM ratio: spread / return-volatility per product

Higher = more profitable per round-trip vs. drift cost.

In [ ]:
rows = []
for cat, names in CATS.items():
    for n in names:
        sym = f'{cat}_{n}'
        s = mid[sym].dropna()
        if len(s) < 100: continue
        ret_std = s.diff().std()
        # spread proxy: typical bid/ask gap from raw rows
        sub = px[(px['product']==sym) & px['bid_price_1'].notna() & px['ask_price_1'].notna()]
        sprd = (sub['ask_price_1'] - sub['bid_price_1']).mean()
        rows.append({'cat': cat, 'sym': sym, 'std': s.std(), 'ret_std': ret_std,
                     'spread': sprd, 'mm_ratio': sprd / ret_std if ret_std > 0 else 0})
tbl = pd.DataFrame(rows).sort_values('mm_ratio', ascending=False)
tbl.round(2)

**SNACKPACK products dominate** with mm_ratio 2.0–2.7 (vs. ~1.0 elsewhere). UV_VISOR, OXYGEN_SHAKE, GALAXY_SOUNDS are decent secondary MM targets.

## 5. Strategy summary going into the algo

1. **PEBBLES basket arb** — for each pebble: fair = 50000 − sum(other 4 mids). Aggressively take when book diverges from this fair, MM around it otherwise. Position limit 10 (tiny), so rapid turnover.
2. **SNACKPACK MM** — anchor each at micro-price; tight std + wide spread (16–18) is the cleanest MM target. CHOCOLATE+VANILLA ≈ 19,940 gives a soft pair constraint to bias fair.
3. **Other categories** — light MM around micro with conservative offsets. Lower edge but free PnL.
